# EVIDENCELOCK: free GPU availability check
This notebook has not been executed in Colab. No package installation, model download, authentication or paid compute is requested.
Choose Runtime > Change runtime type > a free GPU option if offered > Save, then run the cell.
A CUDA tensor check does not establish model, QLoRA or AlignTune readiness.

In [ ]:
import json, platform, subprocess, sys
from pathlib import Path
report = {'python': sys.version, 'platform': platform.platform()}
try:
    p = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free,driver_version', '--format=csv'], capture_output=True, text=True, timeout=15)
    report['gpu'] = p.stdout
    report['gpu_exit_code'] = p.returncode
except Exception as exc:
    report['gpu_error'] = type(exc).__name__
try:
    import torch
    report.update(torch=torch.__version__, cuda_runtime=torch.version.cuda, cuda_available=torch.cuda.is_available())
    if torch.cuda.is_available():
        report['free_total_vram_bytes'] = torch.cuda.mem_get_info()
        report['tensor_check'] = (torch.ones((8,8), device='cuda') @ torch.ones((8,8), device='cuda')).sum().item() == 512
except Exception as exc:
    report['torch_error'] = type(exc).__name__
report['model_training_ready'] = False
print(json.dumps(report, indent=2))
Path('evidencelock-gpu-check.json').write_text(json.dumps(report, indent=2))
